Suppose $\mathcal{M}$ is defined as
$$
\mathcal{M} \coloneqq \left\{x\mapsto a^{(L)}(x): a^{(l+1)}(x) = \varphi^{(l)} \odot \tilde{a}^{(l+1)}(x), \tilde{a}^{(l+1)}(x)=\left(W^{(l)}a^{(l)}(x)+b^{(l)}\right),\varphi^{(l)}\in C^{2}(\mathbb{R};\mathbb{R}), \enspace \text{for} \; l=0,\dots,L-1\right\},
$$
with $a^{(0)}=x$.

For the sake of simplicity we restrict ourselves to $L=1$ and $\varphi^{(1)} = \operatorname{id}$, then for $f\in\mathcal{M}$ we have

$$
f(x) = W^{(1)}\left(\varphi^{(0)} \odot \left(W^{(0)}x+b^{(0)}\right)\right)+b^{(1)}.
$$

We define the vector of parameters

$$
\theta \coloneqq \begin{pmatrix}\operatorname{vec}(W^{(0)},b^{(0)})\\ 
    \operatorname{vec}(W^{(1)},b^{(1)})\end{pmatrix},
$$
which has size $P = n_1(n_0+1) + n_2(n_1+1)$, and let that be corresponding to parameter space $\Theta$, where we define the map
$$
F : \Theta \to \mathcal{M},\enspace F \in C^2(\Theta;\mathcal{M}).
$$


We can of course (for clarity) write down 
$$
\theta = \begin{pmatrix} W^{(0)}_{1,1} & W^{(0)}_{2,1} & \cdots & W^{(0)}_{n_1,n_0} & b^{(0)}_1 & \cdots &  W^{(1)}_{1,1} &  \cdots & b^{(1)}_{n_2}   \end{pmatrix}^T.
$$




Suppose we are interested in $\partial_{\theta^p} F(\theta)$ for every $p\in\{1,\dots,P\}$, we compute all terms:


For $p = 1,\dots,n_1 n_0$:
$$
(\partial_{\theta_p} F(\theta))(x) = (\partial_{W^{(0)}_{i,j}} F(\theta))(x) = W^{(1)}\begin{pmatrix} \mathbb{1}_{\{i=1\}} x_j \dot{\varphi}^{(0)} \left(\tilde{a}^{(1)}_1(x)\right) \\ \vdots \\ \mathbb{1}_{\{i=n_1\}} x_j \dot{\varphi}^{(0)} \left(\tilde{a}^{(1)}_{n_1}(x)\right)\end{pmatrix},
$$
for $p=n_1 n_0+1,\dots, n_1(n_0+1)$:
$$
(\partial_{\theta_p} F(\theta))(x) = (\partial_{b^{(0)}_{i}} F(\theta))(x) = W^{(1)}\begin{pmatrix} \mathbb{1}_{\{i=1\}} \dot{\varphi}^{(0)} \left(\tilde{a}^{(1)}_1(x)\right) \\ \vdots \\ \mathbb{1}_{\{i=n_1\}} \dot{\varphi}^{(0)} \left(\tilde{a}^{(1)}_{n_1}(x)\right)\end{pmatrix},
$$
for $p=n_1(n_0+1)+1,\dots,n_1(n_0+1)+n_2n_1$:
$$
(\partial_{\theta_p} F(\theta))(x) = (\partial_{W^{(1)}_{i,j}} F(\theta))(x) = \begin{pmatrix} \mathbb{1}_{\{i=1\}} a^{(1)}_j(x) \\ \vdots \\ \mathbb{1}_{\{i=n_2\}} a^{(1)}_j(x) \end{pmatrix}
$$
and for $p=n_1(n_0+1)+n_2n_1+1,\dots,n_1(n_0+1)+n_2(n_1+1)$:
$$
(\partial_{\theta_p} F(\theta))(x) = (\partial_{b^{(1)}_{i}} F(\theta))(x) = \begin{pmatrix} \mathbb{1}_{\{i=1\}}  \\ \vdots \\ \mathbb{1}_{\{i=n_2\}} \end{pmatrix}
$$

So in total over $P$ we have:

$$
(D_{\theta} F(\theta))(x) = \begin{pmatrix} (\partial_{W^{(0)}_{1,1}} F(\theta))(x) &  \cdots & (\partial_{b^{(1)}_{n_2}} F(\theta))(x)
\end{pmatrix},
$$
which has size $n_L \times P$.

With the kernel defined as 
$$
K(x,x') = (D_{\theta}F(\theta))(x) ((D_{\theta}F(\theta))(x'))^T
$$

We compute these directly below:

In [61]:
import torch
from torch import nn
import numpy as np
from torch.func import functional_call, jacrev, vmap


In [62]:
def varphi(x):
    return np.tanh(x)

def dot_varphi(x):
    return 1-(np.tanh(x))**2

def t_a(W,b,a):
    return W@a+b

def a(t_a):
    return varphi(t_a)


# TO BE OPTIMIZED


def dW0(W1,t_a1,x):
    n0 = x.shape[0]
    n1 = t_a1.shape[0]
    A = np.zeros(shape=(n1,n1*n0))
    for j in range(n0):
        for i in range(n1):
            A[i,j*n1:(1+j)*n1] = np.eye(n1)[i]*x[j]*dot_varphi(t_a1[i])
    return W1@A

def db0(W1,t_a1):
    n1 = t_a1.shape[0]
    A = np.zeros(shape=(n1,n1))
    for i in range(n1):
        A[i,:] = np.eye(n1)[i]*dot_varphi(t_a1[i])
    return W1@A


def dW1(a1,n2):
    n1 = a1.shape[0]
    A = np.zeros(shape=(n2,n2*n1))
    for i in range(n2):
        for j in range(n1):
            A[i,j*n2:(1+j)*n2] = np.eye(n2)[i]*a1[j]

    return A

def db1(n2):
    return np.eye(n2)

def dF(W1,x,t_a1,a1):
    n0 = x.shape[0]
    n1 = a1.shape[0]
    n2 = W1.shape[0]
    A = np.zeros(shape=(n2,n1*(n0+1)+n2*(n1+1)))
    A[:,0:n1*n0] = dW0(W1,t_a1,x)
    A[:,n1*n0:n1*(n0+1)] = db0(W1,t_a1)
    A[:,n1*(n0+1):n1*(n0+1)+n2*n1] = dW1(a1,n2)
    A[:,n1*(n0+1)+n2*n1:n1*(n0+1)+n2*(n1+1)] = db1(n2)
    return A



In [63]:
x = np.array([-1,1.2])
W0 = np.array([[-0.1,0.1],[1,0.5],[0.8,0]]) # 3x2
b0 = np.array([0.5,0.5,0]) # 3x1
W1 =  np.array([[0.8,1.4,0],[0.1,-0.1,1.2]])# 2x3
b1 = np.array([0,0.2]) # 2x1

a0 = x
t_a1 = t_a(W0,b0,a0)
a1 = a(t_a1)
t_a2 = t_a(W1,b1,a1)
a2 = t_a2

D = dF(W1,x,t_a1,a1)

K = D@np.transpose(D)
print(K)
print(np.linalg.eig(K).eigenvalues)

[[ 9.28528721 -0.36648933]
 [-0.36648933  3.42658576]]
[9.30812383+0.j 3.40374915+0.j]


In [64]:
# Create a net that makes sense

Net = nn.Sequential(nn.Linear(2,3),nn.Tanh(),nn.Linear(3,2)).double()

In [65]:
params = dict(Net.named_parameters())
for name,p in params.items():
    print(name,p.shape)

0.weight torch.Size([3, 2])
0.bias torch.Size([3])
2.weight torch.Size([2, 3])
2.bias torch.Size([2])


In [66]:
with torch.no_grad():
    Net[0].weight.copy_(torch.from_numpy(W0))
    Net[0].bias.copy_(torch.from_numpy(b0))
    Net[2].weight.copy_(torch.from_numpy(W1))
    Net[2].bias.copy_(torch.from_numpy(b1))


In [67]:
xt = torch.tensor((x))
print(xt)
Net(xt)

tensor([-1.0000,  1.2000], dtype=torch.float64)


tensor([ 0.6331, -0.5451], dtype=torch.float64, grad_fn=<ViewBackward0>)

In [68]:
a2

array([ 0.63306263, -0.54511999])

In [69]:
def F(theta,x,model):
    return functional_call(model,theta,x)


In [70]:
DJK = jacrev(F,argnums=0)
theta = dict(Net.named_parameters())
DJK_eval = DJK(theta,xt,Net)
DJK_eval_d = torch.cat([
    DJK_eval[k].transpose(-1, -2).reshape(DJK_eval[k].shape[0], -1)
    if theta[k].ndim == 2
    else DJK_eval[k].reshape(DJK_eval[k].shape[0], -1)
    for k in theta
], dim=1)

D_DJK = DJK_eval_d.detach().numpy()
print(D_DJK)

[[-0.49553833 -1.38609281 -0.          0.594646    1.66331137  0.
   0.49553833  1.38609281  0.          0.6169093   0.          0.09966799
   0.         -0.66403677 -0.          1.          0.        ]
 [-0.06194229  0.09900663 -0.6708662   0.07433075 -0.11880795  0.80503944
   0.06194229 -0.09900663  0.6708662   0.          0.6169093   0.
   0.09966799 -0.         -0.66403677  0.          1.        ]]


In [71]:
print(D)

[[-0.49553833 -1.38609281  0.          0.594646    1.66331137  0.
   0.49553833  1.38609281  0.          0.6169093   0.          0.09966799
   0.         -0.66403677 -0.          1.          0.        ]
 [-0.06194229  0.09900663 -0.6708662   0.07433075 -0.11880795  0.80503944
   0.06194229 -0.09900663  0.6708662   0.          0.6169093   0.
   0.09966799 -0.         -0.66403677  0.          1.        ]]


In [72]:
max_diff = torch.max(torch.abs(DJK_eval_d - torch.as_tensor(D)))
print(max_diff.item())

2.220446049250313e-16


In [73]:
print(np.linalg.norm(D_DJK - D) / np.linalg.norm(D))

6.973775891255959e-17


In [74]:
D_DJK@np.transpose(D_DJK)

array([[ 9.28528721, -0.36648933],
       [-0.36648933,  3.42658576]])

In [75]:
K

array([[ 9.28528721, -0.36648933],
       [-0.36648933,  3.42658576]])

In [76]:
def Compute_DJK(xt,model):
    DJK = jacrev(F,argnums=0)
    theta = dict(model.named_parameters())
    DJK_eval = DJK(theta,xt,model)

    return DJK_eval

In [77]:
SampleNet = nn.Sequential(
    nn.Linear(100,1000),nn.Tanh(),
    nn.Linear(1000,50),nn.Tanh(),
    nn.Linear(50,20),
    ).double()

X = torch.randn(100,dtype=torch.double)

In [78]:
from time import perf_counter
from memory_profiler import memory_usage



In [79]:
torch.set_num_threads(1)

start = perf_counter()

peak, result = memory_usage(
    (Compute_DJK, (X, SampleNet)),
    interval=0.1,
    max_usage=True,
    retval=True
)

print(f"Time: {perf_counter() - start:.4f} s")
print(f"Peak RAM: {peak:.2f} MiB")


Time: 3.5453 s
Peak RAM: 393.18 MiB


In [80]:
torch.set_num_threads(2)

start = perf_counter()

peak, result = memory_usage(
    (Compute_DJK, (X, SampleNet)),
    interval=0.01,
    max_usage=True,
    retval=True
)

print(f"Time: {perf_counter() - start:.4f} s")
print(f"Peak RAM: {peak:.2f} MiB")


Time: 2.5016 s
Peak RAM: 406.89 MiB


In [81]:
torch.set_num_threads(4)

start = perf_counter()

peak, result = memory_usage(
    (Compute_DJK, (X, SampleNet)),
    interval=0.01,
    max_usage=True,
    retval=True
)

print(f"Time: {perf_counter() - start:.4f} s")
print(f"Peak RAM: {peak:.2f} MiB")


Time: 2.4218 s
Peak RAM: 407.17 MiB


In [82]:
torch.set_num_threads(8)

start = perf_counter()

peak, result = memory_usage(
    (Compute_DJK, (X, SampleNet)),
    interval=0.01,
    max_usage=True,
    retval=True
)

print(f"Time: {perf_counter() - start:.4f} s")
print(f"Peak RAM: {peak:.2f} MiB")

Time: 2.5029 s
Peak RAM: 406.93 MiB


In [83]:
torch.set_num_threads(12)

start = perf_counter()

peak, result = memory_usage(
    (Compute_DJK, (X, SampleNet)),
    interval=0.01,
    max_usage=True,
    retval=True
)

print(f"Time: {perf_counter() - start:.4f} s")
print(f"Peak RAM: {peak:.2f} MiB")

Time: 2.5937 s
Peak RAM: 407.24 MiB
